In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
df=pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-detection/refs/heads/master/data.csv')

In [3]:
df.drop(columns=['id','Unnamed: 32'], inplace=True)

In [4]:

x_train, x_test, y_train, y_test=train_test_split(df.iloc[:,1:],df.iloc[:,0], test_size=0.2)

In [5]:
scale=StandardScaler()
x_train=scale.fit_transform(x_train)
x_test=scale.transform(x_test)

In [6]:
encode=LabelEncoder()
y_train=encode.fit_transform(y_train)
y_test=encode.transform(y_test)

In [7]:
# Input features: keep their original 2D shape
x_train = torch.tensor(x_train, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.float32)

# Target labels: reshape into a single column
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

In [8]:
print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test:", x_test.shape)
print("y_test:", y_test.shape)



x_train: torch.Size([455, 30])
y_train: torch.Size([455, 1])
x_test: torch.Size([114, 30])
y_test: torch.Size([114, 1])


In [9]:
import torch.nn as nn

In [10]:
from torch.utils.data import Dataset, DataLoader, TensorDataset

In [11]:
# First lets make custom class
class custom(Dataset):
  def __init__(self, features, labels):
    self.features=features
    self.labels=labels
  def __len__(self):
    return self.features.shape[0]
  def __getitem__(self,index):
    return self.features[index], self.labels[index]

In [12]:
s1=custom(x_train, y_train)

In [13]:
len(s1)

455

In [14]:
s1[101]

(tensor([-0.8868, -0.3812, -0.8595, -0.8074,  0.0366, -0.1318, -0.4594, -0.5410,
         -0.7415,  0.1562, -0.8879,  0.7245, -0.8110, -0.6366,  0.8091,  0.4436,
          0.3700,  0.2628, -0.8894, -0.1169, -0.9479,  0.1346, -0.9045, -0.8128,
          0.4397, -0.0376, -0.2973, -0.2521, -1.0515, -0.0944]),
 tensor([0.]))

In [15]:
# Now inbuilt
data=TensorDataset(x_train,y_train)
loader=DataLoader(
    data,
    batch_size=32,
    shuffle=True
)

In [16]:
class neural_network(nn.Module):

    def __init__(self, num_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(num_features, 24),
            nn.ReLU(),

            nn.Linear(24, 8),
            nn.ReLU(),

            nn.Linear(8, 3),
            nn.ReLU(),

            nn.Linear(3,1),
            nn.Sigmoid()
        )

    def forward(self, features):
        return self.network(features)

In [17]:
s1=neural_network(x_train.shape[1])

In [18]:
crit=nn.BCELoss()
opt=torch.optim.Adam(s1.parameters(), lr=0.03)

In [19]:
for epoch in range(100):
    for batch_no, (X_batch, y_batch) in enumerate(loader):

        # 1. Forward pass
        y_pred = s1(X_batch)

        # 2. Prepare labels
        y_batch = y_batch.float().reshape(-1, 1)

        # 3. Calculate loss
        loss = crit(y_pred, y_batch)

        # 4. Clear previous gradients
        opt.zero_grad()

        # 5. Backpropagation
        loss.backward()

        # 6. Update weights for THIS batch
        opt.step()

        print(
            f"Epoch {epoch+1}, "
            f"Batch {batch_no+1}/{len(loader)}, "
            f"Loss: {loss.item():.4f}"
        )

Epoch 1, Batch 1/15, Loss: 0.7613
Epoch 1, Batch 2/15, Loss: 0.6879
Epoch 1, Batch 3/15, Loss: 0.6565
Epoch 1, Batch 4/15, Loss: 0.5632
Epoch 1, Batch 5/15, Loss: 0.3873
Epoch 1, Batch 6/15, Loss: 0.3011
Epoch 1, Batch 7/15, Loss: 0.2241
Epoch 1, Batch 8/15, Loss: 0.3898
Epoch 1, Batch 9/15, Loss: 0.1931
Epoch 1, Batch 10/15, Loss: 0.1809
Epoch 1, Batch 11/15, Loss: 0.1320
Epoch 1, Batch 12/15, Loss: 0.1308
Epoch 1, Batch 13/15, Loss: 0.4746
Epoch 1, Batch 14/15, Loss: 0.1183
Epoch 1, Batch 15/15, Loss: 0.0016
Epoch 2, Batch 1/15, Loss: 0.2335
Epoch 2, Batch 2/15, Loss: 0.0230
Epoch 2, Batch 3/15, Loss: 0.1957
Epoch 2, Batch 4/15, Loss: 0.0151
Epoch 2, Batch 5/15, Loss: 0.1306
Epoch 2, Batch 6/15, Loss: 0.0436
Epoch 2, Batch 7/15, Loss: 0.0524
Epoch 2, Batch 8/15, Loss: 0.1328
Epoch 2, Batch 9/15, Loss: 0.0412
Epoch 2, Batch 10/15, Loss: 0.0278
Epoch 2, Batch 11/15, Loss: 0.0291
Epoch 2, Batch 12/15, Loss: 0.0399
Epoch 2, Batch 13/15, Loss: 0.0940
Epoch 2, Batch 14/15, Loss: 0.0231
Epo